# SECOM Data Modeling
---

### Imports and creating test-train split

In [2]:
# Imports
from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    average_precision_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier

In [3]:
# Pulling in the SECOM data and loading data into feature and label dataframes
secom = fetch_ucirepo(id=179)
df = pd.DataFrame(secom.data.original)
X = df.drop(columns=["class", "timestamp"])
y = df["class"]
# converting "-1" passing label to "0"
y = y.replace(-1, 0)

In [4]:
# Splitting data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

class
0    0.933759
1    0.066241
Name: proportion, dtype: float64
class
0    0.933121
1    0.066879
Name: proportion, dtype: float64


### Creating custom Secom dataset preprocess transformer

In [6]:
class SecomPreProcessor(BaseEstimator, TransformerMixin):
    def __init__(self, missing_threshold: int, corr_threshold: int):
        self.missing_threshold = missing_threshold
        self.corr_threshold = corr_threshold

    def fit(self, X: pd.DataFrame, y=None):
        X = X.copy()

        # filtering out features with null percentage above threshold
        self.null_cols_ = X.columns[X.isna().mean() > self.missing_threshold].to_list()
        X = X.drop(columns=self.null_cols_)
        
        # imputing NaN with median
        self.medians_ = X.median()
        X = X.fillna(self.medians_)

        # Remove constant variance features
        self.zero_var_cols_ = X.columns[X.var() == 0].to_list()
        X = X.drop(columns=self.zero_var_cols_)

        # dropping columns correlated above threshold
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

        self.corr_cols_ = [
            col
            for col in upper.columns
            if any(upper[col] > self.corr_threshold)
        ]

        X = X.drop(columns=self.corr_cols_)

        self.feature_names_out_ = X.columns.to_list()

        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        X = X.copy()

        X = X.reindex(columns=self.feature_names_out_)

        X = X.fillna(self.medians_)

        return X

    def get_feature_names_out(self, input_features=None) -> np.array:
        return np.array(self.feature_names_out_)

___

### Creating a loop to test the performance of several models at once

In [7]:
models_needs_scaling = {
    "Logistic Regression": LogisticRegression(
        penalty='l1', 
        random_state=42, 
        class_weight="balanced", 
        solver='liblinear', 
        max_iter=5000        
    )
}
models_no_scaling = {
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Random Forest (small)": RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=10,
        min_samples_split=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),
    "Histogram Gradient Boost": HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=4,
        max_iter=300,
        random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        random_state=42,
        eval_metric="logloss"
    )
}

In [8]:
def run_performance(models_needs_scaling: dict, models_no_scaling: dict, X_train : pd.DataFrame = X_train,
                     y_train: pd.DataFrame = y_train) -> pd.DataFrame:
    results = []

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for name, model in models_needs_scaling.items():
        pipe = Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("scale", StandardScaler()),
        ("model", model)
        ])

        scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring={
                "roc_auc": "roc_auc",
                "pr_auc": "average_precision"
            },
            return_train_score=True,
            n_jobs=-1
        )

        results.append({
            "model": name,
            "train_roc_auc": scores["train_roc_auc"].mean(),
            "train_pr_auc": scores["train_pr_auc"].mean(),
            "test_roc_auc_mean": scores["test_roc_auc"].mean(),
            "test_roc_auc_std": scores["test_roc_auc"].std(),
            "test_pr_auc_mean": scores["test_pr_auc"].mean(),
            "test_pr_auc_std": scores["test_pr_auc"].std(),
        })

    for name, model in models_no_scaling.items():
        pipe = Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("model", model)
        ])

        scores = cross_validate(
            pipe,
            X_train,
            y_train,
            cv=cv,
            scoring={
                "roc_auc": "roc_auc",
                "pr_auc": "average_precision"
            },
            return_train_score=True,
            n_jobs=-1
        )

        results.append({
            "model": name,
            "train_roc_auc": scores["train_roc_auc"].mean(),
            "train_pr_auc": scores["train_pr_auc"].mean(),
            "test_roc_auc_mean": scores["test_roc_auc"].mean(),
            "test_roc_auc_std": scores["test_roc_auc"].std(),
            "test_pr_auc_mean": scores["test_pr_auc"].mean(),
            "test_pr_auc_std": scores["test_pr_auc"].std(),
        })

    performance_summary = pd.DataFrame(results).sort_values("test_pr_auc_mean", ascending=False).reset_index(drop=True)
    return performance_summary

In [9]:
performance_summary = run_performance(models_needs_scaling=models_needs_scaling, models_no_scaling=models_no_scaling)
performance_summary

,model,train_roc_auc,train_pr_auc,test_roc_auc_mean,test_roc_auc_std,test_pr_auc_mean,test_pr_auc_std
0,Random Forest (small),1.000000,1.000000,0.724020,0.048940,0.200041,0.052828
1,Random Forest,1.000000,1.000000,0.722766,0.054694,0.198035,0.059456
2,XGBoost,1.000000,1.000000,0.698303,0.063096,0.194247,0.048157
3,Logistic Regression,0.998701,0.970597,0.599604,0.082212,0.153012,0.079866
4,Histogram Gradient Boost,1.000000,1.000000,0.655505,0.061309,0.142040,0.017916


In [13]:
best_pipe = Pipeline([
        ("preprocess", SecomPreProcessor(missing_threshold=0.90, corr_threshold=0.95)),
        ("model", RandomForestClassifier(
        n_estimators=500,
        max_depth=8,
        min_samples_leaf=10,
        min_samples_split=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
        ))
])

In [14]:
def threshold_sweep(best_pipe: Pipeline, thresholds: list[int], X_train: pd.DataFrame = X_train, 
                    y_train: pd.DataFrame = y_train, X_test: pd.DataFrame = X_test, y_test: pd.DataFrame = y_test) -> pd.DataFrame:
    best_pipe.fit(X_train, y_train)
    y_proba = best_pipe.predict_proba(X_test)[:,1]

    results = []

    for threshold in thresholds:
        y_pred = (y_proba > threshold).astype(int)

        results.append({
            "threshold": threshold,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred),
            "f1" : f1_score(y_test, y_pred),
            "flagged": y_pred.sum()
        })
    threshold_df = pd.DataFrame(results).sort_values("precision", ascending=False).reset_index(drop=True)
    return threshold_df

In [15]:
thresholds = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
threshold_df = threshold_sweep(best_pipe=best_pipe, thresholds=thresholds)
threshold_df

,threshold,precision,recall,f1,flagged
0,0.50,0.250000,0.142857,0.181818,12
1,0.40,0.236842,0.428571,0.305085,38
2,0.30,0.142857,0.714286,0.238095,105
3,0.20,0.085470,0.952381,0.156863,234
4,0.10,0.067093,1.000000,0.125749,313
5,0.05,0.066879,1.000000,0.125373,314


In [ ]:
# Sweep threshold probability and print classification report
for threshold in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
    
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\nThreshold = {threshold}")

    print(
        classification_report(
            y_test,
            y_pred,
            digits=3
        )
    )

In [ ]:
# Plot the confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Pass", "Fail"])
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Plot ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve")
plt.show()

In [ ]:
# Plot precision-recall curve
PrecisionRecallDisplay.from_predictions(y_test, y_prob)
plt.title("Precision-Recall Curve")
plt.show()

In [ ]:
# Checking coefficients
coef = pd.Series(
    lr_model.coef_[0],
    index=X_train.columns
)
selected_features = coef[coef != 0].sort_values(key=abs, ascending=False)
print("Number of selected features:", selected_features.shape[0])
print(selected_features.head(30))

In [ ]:
# Plotting top coefficients
top_coef = selected_features.head(20).sort_values()

top_coef.plot(kind="barh", figsize=(8, 6))

plt.xlabel("Coefficient")
plt.title("Top L1 Logistic Regression Coefficients")
plt.show()

## Learnings from LR
1. We tried several different probaility thresholds for prediction, failure class precision is poor and largely unaffected by threshold
2. Even with L1 regularization, we still have ~220 features with non-zero coefficients
    - indicative that importance is spread over sensors or there are interactions present which need non-linear modeling

---

## Training a Random Forest Calssifier on the unscaled training data

In [ ]:
# Creating the RF Classifier
rf = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print("PR-AUC:", average_precision_score(y_test, y_prob_rf))

for t in [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]:
    y_pred_rf = (y_prob_rf >= t).astype(int)
    print(f"\nThreshold = {t}")
    print(classification_report(y_test, y_pred_rf, digits=3))

In [ ]:
# Checking feature importances
importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print(importance.head(30))

In [ ]:
importance["cumulative"] = importance["importance"].cumsum()

print(
    importance.loc[
        importance["cumulative"] <= 0.5
    ].shape[0]
)

In [ ]:
result = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=20,
    random_state=42,
    scoring="roc_auc"
)

perm_importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance": result.importances_mean
})

perm_importance.sort_values(
    "importance",
    ascending=False
).head(20)

In [ ]:
rf_train_prob = rf.predict_proba(X_train)[:,1]

print(
    roc_auc_score(y_train, rf_train_prob)
)

In [ ]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=4,
    max_iter=300,
    random_state=42
)

hgb.fit(X_train, y_train)

y_prob_hgb = hgb.predict_proba(X_test)[:, 1]

print("Test ROC-AUC:",
      roc_auc_score(y_test, y_prob_hgb))

print("Test PR-AUC:",
      average_precision_score(y_test, y_prob_hgb))

In [ ]:
train_prob_hgb = hgb.predict_proba(X_train)[:,1]

print(
    "Train ROC-AUC:",
    roc_auc_score(y_train, train_prob_hgb)
)

## Training an XGBoost Classifier

In [ ]:
# Training an XGBoost Classifier
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))
print("PR-AUC:", average_precision_score(y_test, y_prob_xgb))

In [ ]:
xgb_train_prob = xgb.predict_proba(X_train)[:,1]

print(
    roc_auc_score(y_train, xgb_train_prob)
)

### Adding cross-validation

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

scores = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print(scores)
print("Mean:", scores.mean())
print("Std:", scores.std())

In [ ]:
# Smaller rf
rf_small = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=10,
    min_samples_split=20,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_small.fit(X_train, y_train)

y_prob_rf = rf_small.predict_proba(X_test)[:, 1]
y_prob_rf_train = rf_small.predict_proba(X_train)[:, 1]

print("ROC-AUC (Test):", roc_auc_score(y_test, y_prob_rf))
print("PR-AUC (Test):", average_precision_score(y_test, y_prob_rf))
print("ROC-AUC (Train):", roc_auc_score(y_train, y_prob_rf_train))

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_val_score(
    rf_small,
    X_train,
    y_train,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

print(scores)
print("Mean:", scores.mean())
print("Std:", scores.std())